# XN Benchmark: BioMiner and MolScribe on Colab

Runs the full pipeline from this repo on a Colab GPU runtime: setup, inference, metrics, and results.
The BioVista images and Jirapa's DECIMER results are downloaded automatically.

**Before you start**
- Runtime: GPU. An L4 or A100 is recommended, since BioMiner runs three model servers at once.
- Google Drive folder (set `DATA_DIR` below) containing:
  - `course_papers/` with the six course PDFs and the ground truth CSV from Canvas
  - `decimer_crops/` with Jirapa's DECIMER crops of those papers (if missing, only that step is skipped)
- Colab secrets (key icon in the left sidebar): `BIOMINER_API_KEY`, `BIOMINER_BASE_URL`,
  `BIOMINER_TEXT_MODEL`, `BIOMINER_VISION_MODEL`

## 1. Install conda
The runtime restarts automatically after this cell. Then continue with the next cell.

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:06
🔁 Restarting kernel...


## 2. Settings
Set `DATA_DIR` to your Google Drive data folder.

In [1]:
REPO_URL = "https://github.com/Showrish/xn-ocsr-benchmark-1.2.git"
DATA_DIR = "/content/drive/MyDrive/xn_data"

## 3. Get the code and link the data

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
REPO_DIR = "/content/xn-ocsr-benchmark"
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls

Mounted at /content/drive
/content/xn-ocsr-benchmark
config	notebooks  requirements.txt  run_pipeline.py  src
envs	README.md  results	     scripts	      tests


## 4. Download BioMiner, the BioVista data, model weights, and Jirapa's DECIMER results

In [3]:
!pip install -q -r requirements.txt
!python scripts/setup_external.py

Cloning into '/content/xn-ocsr-benchmark/external/BioMiner'...
remote: Enumerating objects: 1019, done.
remote: Counting objects: 100% (1019/1019), done.
remote: Compressing objects: 100% (386/386), done.
remote: Total 1019 (delta 627), reused 993 (delta 625), pack-reused 0 (from 0)
Receiving objects: 100% (1019/1019), 114.40 MiB | 20.09 MiB/s, done.
Resolving deltas: 100% (627/627), done.
Updating files: 100% (977/977), done.
19:18:08 | INFO | setup | Downloading BioVista bioactivity_triplet_extraction
Downloading...
From: https://drive.google.com/uc?id=14icAlg_jEy3uE7FOcDD-EYn3UIKQxC__
To: /content/xn-ocsr-benchmark/external/BioMiner/BioVista/bioactivity_extraction_normalization.tar.gz
100% 1.07M/1.07M [00:00<00:00, 155MB/s]
19:18:15 | INFO | setup | Unpacked bioactivity_extraction_normalization.tar.gz
19:18:15 | INFO | setup | Downloading BioVista structure_bioactivity_annotation
Downloading...
From (original): https://drive.google.com/uc?id=1cLC3NePtxctIurnCVaNLw0JzBWYrWTVW
From (r

## 5. Create one environment per tool
The exact package versions are exported to `envs/lock/` for reproducibility.

In [4]:
!conda env create -q -f envs/molscribe.yml
!conda env create -q -f envs/scoring.yml
!mkdir -p envs/lock
!conda env export -n xn-molscribe > envs/lock/xn-molscribe.lock.yml
!conda env export -n xn-scoring > envs/lock/xn-scoring.lock.yml

Retrieving notices: ...working... done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done


In [8]:
!conda run -n xn-molscribe python -c "import torch, molscribe; print('MolScribe OK, GPU:', torch.cuda.is_available())"

MolScribe OK, GPU: True
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/onmt/modules/sparse_activations.py:48: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, dim=0):
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/onmt/modules/sparse_activations.py:68: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/onmt/modules/sparse_losses.py:13: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  

In [7]:
import os
os.environ["MPLBACKEND"] = "Agg"

## 6. Run the Tests


In [9]:
!conda run -n xn-scoring python -m pytest -q tests

.........                                                                [100%]
9 passed in 0.43s


## 8. Run the Benchmark

In [10]:
!python run_pipeline.py --steps ocsr_molscribe evaluate

19:30:29 | INFO | pipeline | Starting step 'ocsr_molscribe': conda run --no-capture-output -n xn-molscribe python -m src.run_molscribe --dataset biovista_ocsr --config /content/xn-ocsr-benchmark/config/config.yaml
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/onmt/modules/sparse_activations.py:48: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, dim=0):
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/onmt/modules/sparse_activations.py:68: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/usr/local/envs/xn-molscribe/lib/python3.10

## 9. Results

In [11]:
!cat results/summary.md

cat: results/summary.md: No such file or directory


In [13]:
%cd /content/xn-ocsr-benchmark
!git pull -q
!python run_pipeline.py --steps ocsr_molscribe evaluate

/content/xn-ocsr-benchmark
19:46:12 | INFO | pipeline | Starting step 'ocsr_molscribe': conda run --no-capture-output -n xn-molscribe python -m src.run_molscribe --dataset biovista_ocsr --config /content/xn-ocsr-benchmark/config/config.yaml
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/onmt/modules/sparse_activations.py:48: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, dim=0):
/usr/local/envs/xn-molscribe/lib/python3.10/site-packages/onmt/modules/sparse_activations.py:68: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/usr/local/envs/

## 10. Download results to commit them
Upload the contents to the repo's `results/` and `envs/lock/` folders.

In [17]:
%cd /content/xn-ocsr-benchmark
!git pull -q
!python run_pipeline.py --steps evaluate
!cat results/summary.md
!zip -qr results_and_locks.zip results envs/lock -x "results/raw/*"
from google.colab import files
files.download("results_and_locks.zip")

/content/xn-ocsr-benchmark
20:07:09 | INFO | pipeline | Starting step 'evaluate': conda run --no-capture-output -n xn-scoring python -m src.evaluate --config /content/xn-ocsr-benchmark/config/config.yaml
20:07:11 | INFO | evaluate | Loaded predictions_molscribe.csv (100 rows)
20:07:11 | INFO | evaluate | Loaded predictions_decimer.csv (100 rows)
20:07:12 | INFO | evaluate | Scored 2 tools on 100 BioVista images
20:07:12 | INFO | evaluate | No course-paper predictions found, skipping the course evaluation
20:07:12 | INFO | evaluate | Wrote results to /content/xn-ocsr-benchmark/results
20:07:12 | INFO | pipeline | Step 'evaluate' finished
20:07:12 | INFO | pipeline | Summary: evaluate=ok
# Benchmark Results

## BioVista OCSR images (100 images)

Share of images whose prediction exactly matches the correct SMILES after RDKit canonicalization.
Published values are from the BioMiner README for all 8,861 images.

| tool | group | images | exact_match | exact_match_no_stereo | valid_rate | me

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
!rm -rf /tmp/check && git clone -q https://github.com/Showrish/xn-ocsr-benchmark-1.2.git /tmp/check
%cd /tmp/check
!git log --oneline | head -3
!ls -a
!ls results results/biovista_ocsr envs/lock
!grep -c "Key results" README.md
!grep -c "authors' own rule" src/biovista_ocsr.py
!grep -c "TOOL_NAMES" src/evaluate.py
!grep -c "xn-ocsr-benchmark-1.2" notebooks/run_on_colab.ipynb
%cd /content/xn-ocsr-benchmark

/tmp/check
669e8a4 Delete results/envs/lock directory
017d307 Delete results/results directory
f9b3557 Add files via upload
.	.env.example  .gitignore  requirements.txt  scripts
..	envs	      notebooks   results	    src
config	.git	      README.md   run_pipeline.py   tests
envs/lock:
README.md  xn-molscribe.lock.yml  xn-scoring.lock.yml

results:
biovista_ocsr  README.md  summary.json	summary.md

results/biovista_ocsr:
comparison.csv	      metrics_per_image.csv	 sample_manifest.csv
metrics_by_group.csv  predictions_molscribe.csv
1
1
2
1
/content/xn-ocsr-benchmark
